In [1]:
import os
import sys

# 设置缓存目录路径
cache_dir = r'F:\Anaconda\envs\pytorch\cache'

# 确保目录存在
os.makedirs(cache_dir, exist_ok=True)

# 设置所有相关环境变量
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HUGGINGFACE_HUB_CACHE'] = cache_dir

print(f"设置的缓存目录: {cache_dir}")

# 现在导入transformers
from transformers import pipeline
from transformers.utils import TRANSFORMERS_CACHE

print(f"实际缓存路径: {TRANSFORMERS_CACHE}")

# 验证路径是否正确
if cache_dir == TRANSFORMERS_CACHE:
    print("✅ 缓存路径设置成功！")
else:
    print(f"❌ 缓存路径设置失败！期望: {cache_dir}, 实际: {TRANSFORMERS_CACHE}")

# 运行情感分析
classifier = pipeline("sentiment-analysis")
result = classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

print(f"\n分析结果: {result}")

# 检查缓存目录是否创建了文件
print(f"\n缓存目录内容:")
if os.path.exists(cache_dir):
    items = os.listdir(cache_dir)
    if items:
        for item in items:
            item_path = os.path.join(cache_dir, item)
            if os.path.isdir(item_path):
                print(f"  📁 {item}")
            else:
                print(f"  📄 {item}")
    else:
        print("   (空目录)")
else:
    print("   (目录不存在)")

设置的缓存目录: F:\Anaconda\envs\pytorch\cache


f:\Anaconda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
f:\Anaconda\envs\pytorch\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


实际缓存路径: F:\Anaconda\envs\pytorch\cache
✅ 缓存路径设置成功！


Device set to use cuda:0



分析结果: [{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

缓存目录内容:
  📁 .locks
  📁 datasets
  📁 datasets--glue
  📁 evaluate
  📁 metrics
  📁 models--bert-base-uncased
  📁 models--distilbert--distilbert-base-uncased-finetuned-sst-2-english
  📁 modules
  📁 xet


PyTorch已经帮我们自动实现了计算图生成和自动求梯度的功能。

In [2]:
import torch

x = torch.tensor(1.0, requires_grad=True) #指定需要计算梯度
y = torch.tensor(1.0, requires_grad=True) #指定需要计算梯度
v = 3*x+4*y
u = torch.square(v)
z = torch.log(u)

z.backward() #反向传播求梯度

print("x grad:", x.grad)
print("y grad:", y.grad)

x grad: tensor(0.8571)
y grad: tensor(1.1429)


利用PyTorch里的自动求梯度的能力，可以大大简化我们利用梯度下降方法对模型的训练。

用PyTorch实现线性回归

In [3]:
import torch

# 确保CUDA可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 生成数据
inputs = torch.rand(100, 3) # 随机生成shape为(100,3)的tensor，里边每个元素的值都是0-1之间
weights = torch.tensor([[1.1], [2.2], [3.3]]) #预设的权重
bias = torch.tensor(4.4) #预设的bias
targets = inputs @ weights + bias + 0.1*torch.randn(100, 1) #增加一些误差，模拟真实情况

In [4]:
# 初始化参数时直接放在CUDA上，并启用梯度追踪
w = torch.rand((3, 1), requires_grad=True, device=device)
b = torch.rand((1,), requires_grad=True, device=device)

In [ ]:
# 将数据移至相同设备.to(device)
inputs = inputs.to(device)
targets = targets.to(device)

#设置超参数
epoch = 10000
lr = 0.003

for i in range(epoch):
    outputs = inputs @ w + b
    loss = torch.mean(torch.square(outputs - targets))
    print("loss:", loss.item())

    loss.backward()

    with torch.no_grad(): #下边的计算不需要跟踪梯度
        w -= lr * w.grad
        b -= lr * b.grad

    # 清零梯度 防止错误累计
    w.grad.zero_()
    b.grad.zero_()

print("训练后的权重 w:", w)
print("训练后的偏置 b:", b)

loss: 43.38568115234375
loss: 42.51448059082031
loss: 41.66090774536133
loss: 40.82460021972656
loss: 40.00520706176758
loss: 39.20238494873047
loss: 38.415809631347656
loss: 37.6451416015625
loss: 36.89006423950195
loss: 36.1502571105957
loss: 35.425418853759766
loss: 34.71523666381836
loss: 34.01942443847656
loss: 33.337684631347656
loss: 32.669734954833984
loss: 32.015289306640625
loss: 31.374086380004883
loss: 30.745849609375
loss: 30.130321502685547
loss: 29.527244567871094
loss: 28.93636131286621
loss: 28.357431411743164
loss: 27.79020881652832
loss: 27.23446273803711
loss: 26.689950942993164
loss: 26.15645408630371
loss: 25.633745193481445
loss: 25.121610641479492
loss: 24.619829177856445
loss: 24.12820053100586
loss: 23.646509170532227
loss: 23.174562454223633
loss: 22.712158203125
loss: 22.259105682373047
loss: 21.815214157104492
loss: 21.38030242919922
loss: 20.954179763793945
loss: 20.53667640686035
loss: 20.12761878967285
loss: 19.726825714111328
loss: 19.334142684936523
lo

在模型训练过程中，有一个最重要的指标，就是loss值。而通过观察loss值变化的曲线，可以得到很多有用的信息。

In [ ]:
import torch
from torch.utils.tensorboard import SummaryWriter

# 确保CUDA可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 生成数据
inputs = torch.rand(100, 3)  # 生成shape为(100,3)的tensor，里边每个元素的值都是0-1之间
weights = torch.tensor([[1.1], [2.2], [3.3]])
bias = torch.tensor(4.4)
targets = inputs @ weights + bias + 0.1 * torch.randn(100, 1)  # 增加一些随机，模拟真实情况

# 创建一个SummaryWriter实例   指定目录产生文件
writer = SummaryWriter(log_dir="F:/VS code project/AI/Learning/PyTorch/runs/")

# 初始化参数时直接放在CUDA上，并启用梯度追踪
w = torch.rand(3, 1, requires_grad=True, device=device)
b = torch.rand(1, requires_grad=True, device=device)

# 将数据移至相同设备
inputs = inputs.to(device)
targets = targets.to(device)

epoch = 10000
lr = 0.003

for i in range(epoch):
    outputs = inputs @ w + b
    loss = torch.mean(torch.square(outputs - targets))
    print("loss:", loss.item())
    #记录loss，三个参数分别：tag，loss值，第几步
    writer.add_scalar("loss/train", loss.item(), i)
    loss.backward()

    with torch.no_grad():  # 下边的计算不需要跟踪梯度
        w -= lr * w.grad
        b -= lr * b.grad

    # 清零梯度
    w.grad.zero_()
    b.grad.zero_()

print("训练后的权重 w:", w)
print("训练后的偏置 b:", b)

loss: 43.799407958984375
loss: 42.84531021118164
loss: 41.912078857421875
loss: 40.999267578125
loss: 40.106414794921875
loss: 39.233097076416016
loss: 38.37887954711914
loss: 37.54335021972656
loss: 36.72609329223633
loss: 35.92671203613281
loss: 35.14482116699219
loss: 34.380027770996094
loss: 33.631961822509766
loss: 32.90026092529297
loss: 32.18456268310547
loss: 31.484519958496094
loss: 30.799789428710938
loss: 30.130033493041992
loss: 29.474931716918945
loss: 28.834152221679688
loss: 28.2073917388916
loss: 27.594337463378906
loss: 26.99469757080078
loss: 26.408164978027344
loss: 25.83446502685547
loss: 25.2733097076416
loss: 24.72443389892578
loss: 24.187557220458984
loss: 23.66242218017578
loss: 23.148773193359375
loss: 22.6463623046875
loss: 22.15493392944336
loss: 21.674257278442383
loss: 21.204092025756836
loss: 20.74420928955078
loss: 20.294384002685547
loss: 19.854394912719727
loss: 19.424028396606445
loss: 19.003076553344727
loss: 18.59132957458496
loss: 18.18858528137207


tensorboard --logdir="F:/VS code project/AI/Learning/PyTorch/runs/"终端输入后
网站查看http://localhost:6006 

这是一个非常理想的loss曲线，前边一段训练过程loss快速下降，到后边loss逐渐变化很小，证明已经收敛。整个训练过程loss变化也很平稳，因为整个曲线很平滑。

上边的loss曲线很完美，但是情况并不总是如此。如果你发现你的loss曲线并不下降，那么有以下几种可能。1.你的代码有bug，你需要检查你的代码。2. 你的学习率设置太大。你可以先将学习率设置成足够小的值，然后看loss值是否下降。如果loss值还是不下降，那么很大可能就是代码的问题了。

根据loss曲线，你也可以判断什么时候终止训练，那就是loss几乎不变，看起来是一条平行于x轴的直线。这时我们认为模型收敛了，梯度下降已经不能进一步减少损失了。此时，我们就得到了最优的模型。